# Train Diabetes Model
Notebook này huấn luyện nhiều mô hình, chọn mô hình tốt nhất theo F1-score, rồi xuất `best_model.pkl` và `scaler.pkl`.

In [1]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def resolve_paths():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / '..' / 'data' / 'diabetes.csv',
        cwd / 'backend' / 'ml' / 'data' / 'diabetes.csv',
        cwd / 'ml' / 'data' / 'diabetes.csv',
    ]
    data_path = None
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate.exists():
            data_path = candidate
            break

    if data_path is None:
        raise FileNotFoundError('Không tìm thấy file diabetes.csv')

    ml_dir = data_path.parent.parent
    models_dir = ml_dir / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)
    return data_path, models_dir

data_path, models_dir = resolve_paths()
print('Data path:', data_path)
print('Models dir:', models_dir)

Data path: D:\Workspace\Python\Diabetes-Diagnosis-app\backend\ml\data\diabetes.csv
Models dir: D:\Workspace\Python\Diabetes-Diagnosis-app\backend\ml\models


In [2]:
df = pd.read_csv(data_path)
print('Shape:', df.shape)
display(df.head())

X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
models = {
    'random_forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'gradient_boosting': GradientBoostingClassifier(random_state=42),
    'logistic_regression': LogisticRegression(max_iter=2000, random_state=42),
}

results = []
best_name = None
best_model = None
best_f1 = -1.0

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
    }
    results.append(metrics)

    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_name = name
        best_model = model

results_df = pd.DataFrame(results).sort_values(by='f1', ascending=False)
display(results_df)
print(f'Best model: {best_name} (F1={best_f1:.4f})')

,model,accuracy,precision,recall,f1
0,random_forest,0.753247,0.660000,0.611111,0.634615
1,gradient_boosting,0.753247,0.666667,0.592593,0.627451
2,logistic_regression,0.714286,0.608696,0.518519,0.560000


Best model: random_forest (F1=0.6346)


In [4]:
best_model_path = models_dir / 'best_model.pkl'
scaler_path = models_dir / 'scaler.pkl'

joblib.dump(best_model, best_model_path)
joblib.dump(scaler, scaler_path)

print('Saved:', best_model_path)
print('Saved:', scaler_path)

Saved: D:\Workspace\Python\Diabetes-Diagnosis-app\backend\ml\models\best_model.pkl
Saved: D:\Workspace\Python\Diabetes-Diagnosis-app\backend\ml\models\scaler.pkl
